<a href="https://colab.research.google.com/github/prad1971/deployment_sample/blob/master/AgenticRAG_MultipleDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-openai langchain-community chromadb pypdf streamlit tiktoken

In [ ]:
import os
import shutil
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
# Create some sample data
folders = [
    "data/hr",
    "data/tech",
    "data/finance"
]

for folder in folders:
  os.makedirs(folder, exist_ok = True)


In [ ]:
# HR Data
with open("data/hr/hr_policy.text","w") as f:
  f.write("""
    1. Employees must maintain professionalism, respect, and ethical conduct at all times in the workplace.
    2. Attendance and punctuality are mandatory; repeated absenteeism or late arrival may lead to disciplinary action.
    3. Harassment, discrimination, and workplace bullying of any kind are strictly prohibited.
    4. Company confidential information must not be shared with unauthorized persons or external parties.
    5. Employees are required to follow all company safety, data security, and compliance guidelines.
  """)


# Tech Dat a

with open("data/tech/tech_imp_docs.text","w") as f:
  f.write("""
    1.Modern GenAI systems are evolving toward multi-modal foundation models that unify text, vision, audio, video, and structured data within a shared latent embedding space for cross-domain understanding.
    2.Agentic AI architectures now leverage multi-agent orchestration frameworks, enabling autonomous task decomposition, tool invocation, memory management, and iterative planning using LLM-powered reasoning.
    3.Enterprises are adopting Agentic RAG (Retrieval-Augmented Generation) pipelines with vector databases, context graphs, and adaptive memory layers to improve long-horizon reasoning and reduce hallucinations.
    4.The industry is shifting toward small domain-specific language models (SLMs) and edge-deployable inference stacks to optimize latency, privacy, and compute efficiency in real-time AI systems.
    5.AI governance is becoming a core engineering layer, with emerging standards around agent observability, AI safety alignment, FinOps for AI workloads, and autonomous policy enforcement frameworks.
  """)

  # Finance data

with open("data/finance/finance_docs.text","w") as f:
  f.write("""
    1.Tesla reported FY2025 revenue of approximately $94.8 billion, representing a ~3% year-over-year decline mainly due to EV pricing pressure and softer automotive margins.
    2.Net income declined sharply to about $3.8 billion, down ~46% YoY, while diluted EPS dropped to nearly $1.08, reflecting rising R&D, AI infrastructure, and operating costs.
    3.Tesla maintained strong liquidity with over $40 billion in cash and short-term investments, supporting continued investments in autonomous driving, robotics, and manufacturing expansion.
    4.The Energy Generation & Storage business emerged as Tesla's fastest-growing segment, generating roughly $12.7 billion revenue with stronger margins compared to the automotive division.
    5.Despite profitability pressure, Tesla generated approximately $6.2 billion in free cash flow in FY2025, highlighting operational cash resilience even amid heavy capital expenditure and strategic investments.
  """)



In [ ]:
# Vector DB Registry
VECTOR_DB_REGISTRY = {
    "hr":{
        "description":"Workplace Professionalism,Attendance Compliance,Anti-Harassment Policy,Confidential Data Protection,Safety & Compliance Standards",
        "path": "vectordb/hr_db"
    },
    "tech":{
        "description":"Multimodal Foundation Models,Multi-Agent Orchestration,Agentic RAG Pipelines,Domain-Specific SLMs,AI Observability & Governance",
        "path": "vectordb/tech_db"
    },
    "finance":{
        "description":"Automotive Revenue Decline,Energy Storage Growth,Free Cash Flow,AI Infrastructure Investment,Liquidity Reserves",
        "path": "vectordb/finance_db"
    }
}

In [ ]:
VECTOR_DB_REGISTRY["finance"]["path"]

'vectordb/finance_db'

In [ ]:
embedding_models = OpenAIEmbeddings()

router_llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

answer_llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50
)

In [ ]:
#Ingestion
def ingest_folder(folder_path,vector_db_path):
  documents = []
  for file in os.listdir(folder_path):
    full_path = os.path.join(folder_path,file)
    # Skip directories, only load files
    if not os.path.isfile(full_path):
      continue
    loader = TextLoader(full_path)
    loaded_docs = loader.load()
    documents.extend(loaded_docs)

  print(f"loaded {len(documents)} documents")

  chunks = splitter.split_documents(documents)

  print(f"Created {len(chunks)} chunks")

  #if os.path.exist(vector_db_path):
  #  shutil.rmtree(vector_db_path)

  vector_db = Chroma.from_documents(
      documents = chunks,
      embedding = embedding_models,
      persist_directory = vector_db_path
  )

  vector_db.persist()
  print(f" vector DB stored at : {vector_db_path}")

In [ ]:
def ingest_vectordb_registry():
  ingest_folder(
      folder_path="data/hr",
      vector_db_path = VECTOR_DB_REGISTRY["hr"]["path"]
  )

  ingest_folder(
      folder_path="data/tech",
      vector_db_path = VECTOR_DB_REGISTRY["tech"]["path"]
  )

  ingest_folder(
      folder_path="data/finance",
      vector_db_path = VECTOR_DB_REGISTRY["finance"]["path"]
  )

In [ ]:
# What makes this system Agentic ( Dynamic )
def route_query(user_query):
    registry_information = ""

    # Build registry information once
    for db_name, db_info in VECTOR_DB_REGISTRY.items():
        registry_information += f"""
        DATABASE NAME: {db_name}
        DATABASE DESCRIPTION: {db_info['description']}
        """

    routing_prompt = f"""
    You are an intelligent routing agent expert in routing user queries
    to the correct vector database.

    Your job is to select the BEST database for the given user question.

    AVAILABLE DATABASES:
    {registry_information}

    USER QUESTION:
    {user_query}

    RULES:
    - Return ONLY the database name
    - No explanation
    - Choose only from: hr, tech, finance, none
    """

    response = router_llm.invoke(routing_prompt)

    selected_database = response.content.strip().lower()

    return selected_database

In [ ]:
def load_vector_database(database_name):
  database_path = VECTOR_DB_REGISTRY[database_name]["path"]
  vectordb = Chroma(
      persist_directory = database_path,
      embedding_function = embedding_models
  )
  return vectordb

In [ ]:
# Retrieval
def retrieve_documents(user_query,database_name):
  print(database_name)
  vectordb = load_vector_database(database_name)
  retriever = vectordb.as_retriever(
      search_kwarg={"k":3}
  )
  retrieved_docs = retriever.invoke(user_query)
  return retrieved_docs

In [ ]:
def generate_answer(user_query,retrieved_docs):
  context = "\n\n".join([
      doc.page_content
      for doc in retrieved_docs
  ])
  final_prompt = f"""
  You are a AI Consistent

  Answer the question using only provided context

  CONTEXT
  {context}

  QUESTION
  {user_query}
  """

  response = answer_llm.stream(final_prompt)
  return response

In [ ]:
def ask_agentic_rag(user_query):

  selected_database = route_query(user_query)

  # Give me chunks from the selected database
  retrieved_docs = retrieve_documents(
      user_query = user_query,
      database_name = selected_database
  )

  # Standara generation
  response_rag = generate_answer(
      user_query = user_query,
      retrieved_docs = retrieved_docs
  )


  for c in response_rag:
    print(c.content,end="", flush = True)

In [ ]:
ingest_vectordb_registry()

loaded 1 documents
Created 3 chunks


/tmp/ipykernel_13151/2570320989.py:25: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


 vector DB stored at : vectordb/hr_db
loaded 1 documents
Created 5 chunks
 vector DB stored at : vectordb/tech_db
loaded 1 documents
Created 3 chunks
 vector DB stored at : vectordb/finance_db


In [ ]:
ask_agentic_rag("what is the Agentic AI")

tech
Agentic AI refers to AI architectures that leverage multi-agent orchestration frameworks, enabling autonomous task decomposition, tool invocation, memory management, and iterative planning using LLM-powered reasoning.
